# 06_02 - Decision Engine: RL Prototype
## 1. Methodology Overview

This notebook builds and applies the experimental RL policy using the same policy inputs as the heuristic engine.
The implementation is intentionally lightweight (tabular Q-learning) and designed for transparent diagnostics.

**Source data used in this notebook:**
- `data/processed/train_features.csv`
- `data/processed/validation_features.csv`

**Core modules used in this notebook:**
- `src/models/train_model.py`
- `src/decision/policy_inputs.py`
- `src/rl/train_rl_agent.py`
- `src/decision/rl_policy.py`

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

train_df = pd.read_csv(project_root / 'data/processed/train_features.csv')
validation_df = pd.read_csv(project_root / 'data/processed/validation_features.csv')

print(f'Train features: {train_df.shape[0]} rows x {train_df.shape[1]} columns')
print(f'Validation features: {validation_df.shape[0]} rows x {validation_df.shape[1]} columns')
display(validation_df.head())

## 2. Prepare Policy Inputs and Train the Q-Learning Agent

The RL agent is trained on policy inputs (forecast + market state) and learns to select one of three actions.
We keep training diagnostics visible to inspect learning quality.

In [ ]:
from src.models.train_model import train_quantile_suite
from src.decision.policy_inputs import prepare_policy_inputs
from src.rl.train_rl_agent import train_q_learning_agent

quantile_output = train_quantile_suite(train_df=train_df, eval_df=validation_df)
policy_inputs_df = prepare_policy_inputs(validation_df, quantile_output.results)

rl_training_artifacts = train_q_learning_agent(policy_inputs_df)

display(rl_training_artifacts.rewards_summary_df)
display(rl_training_artifacts.rewards_history_df.tail(10))

print(f'Learned Q-table states: {len(rl_training_artifacts.agent.q_table)}')

## 3. Apply RL Policy and Inspect Actions

We now generate row-level RL decisions and inspect both the action mix and Q-value diagnostics.

In [ ]:
from src.decision.rl_policy import apply_rl_policy

rl_policy_artifacts = apply_rl_policy(
    agent=rl_training_artifacts.agent,
    policy_inputs_df=policy_inputs_df,
    include_q_values=True,
 )

display(rl_policy_artifacts.decisions_df.head(12))
display(rl_policy_artifacts.action_summary_df)

if rl_policy_artifacts.q_values_df is not None:
    display(rl_policy_artifacts.q_values_df.head(12))